Start Ingestion

In [0]:
file_path = "/Volumes/workspace/mlbb/mlbb_source_files/tournament_data.csv";

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, TimestampType, LongType, DoubleType, DateType

In [0]:
schema = StructType([
  StructField("tournament_code", IntegerType(), False),
  StructField("tournament_name", StringType(), False),
  StructField("tier", StringType(), False),
  StructField("start_date", StringType(), False),
  StructField("end_date", StringType(), False),
  StructField("patch_code", StringType(), True),
  StructField("url", StringType(), True),
  StructField("last_updated", StringType(), True),
  StructField("csv_name", StringType(), True)
])


In [0]:
df = spark.read.csv(path = file_path, header=True, schema=schema)

In [0]:
#display(df.limit(10))

In [0]:
#Write to Bronze table
df.write.format("delta").mode("overwrite").saveAsTable("mlbb.tournament_data_bronze")

**Start transformations**

In [0]:
bronze = spark.read.table("mlbb.tournament_data_bronze");

In [0]:
from pyspark.sql.functions import to_date,col, date_diff

In [0]:

bronze_date_parsed = bronze.withColumn("start_date", to_date( col("start_date"), 'yyyyMMdd') ) \
                            .withColumn("end_date", to_date( col("end_date"), 'yyyyMMdd') ) \
                            .withColumn("last_updated", to_date( col("last_updated"), 'yyyyMMdd') ) ;

In [0]:
bronze_date_parsed = bronze_date_parsed.withColumn("duration_days", date_diff(col("end_date"), col("start_date")) );

In [0]:
#filter out bad data to quartined
bad = bronze_date_parsed.filter( (col("duration_days") <= 0) \
    | (col("duration_days").isNull())
    );
bad.write.format("delta").mode("overwrite").saveAsTable("mlbb.tournament_data_quarantine");

In [0]:
good = bronze_date_parsed.filter(col("duration_days") > 0 );
good.write.format("delta").mode("overwrite").saveAsTable("mlbb.tournament_data_silver");

In [0]:
#display(good.limit(5))

In [0]:
silver_table_name = "mlbb.tournament_data_silver";
silver = spark.read.table(silver_table_name);

In [0]:
from pyspark.sql.functions import count, avg, col, max as _max, min as _min,round;

In [0]:
gold = silver.groupBy("tier").agg(
    count("tournament_code").alias("number_of_tournaments"),
    avg("duration_days").alias("average_duration_days"),
    _max("duration_days").alias("longest_tournament"),
    _min("duration_days").alias("shortest_tournament")
)

In [0]:
gold = gold.withColumn("average_duration_days", round(col("average_duration_days"), 0));

In [0]:
%sql
--select * from mlbb.tournament_data_silver where duration_days <= 0 or duration_days is null;
--select * from mlbb.tournament_data_gold

In [0]:
gold.write.format("delta").mode("overwrite").saveAsTable("mlbb.tournament_data_gold");

In [0]:
#display(gold.limit(5))